# PUL7SAR Phase 18 — Golden Visual GPU Proof
This notebook runs the existing Phase 18 `$0-local` pipeline on the notebook GPU. It does not use a paid image API. Select a GPU runtime before running. The first run executes only candidate 1 through the same sequential batch path used by the final four-candidate proof.

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout)
assert gpu.returncode == 0, 'GPU runtime is required'


In [ ]:
!git clone --depth 1 --branch phase18/story-intelligence https://github.com/pulsar7official/pul7sar-bot.git
%cd pul7sar-bot
!python -m pip install -U -r requirements-phase18-gpu.txt


In [ ]:
!PYTHONPATH=. python tools/phase18_local_readiness.py


In [ ]:
!PYTHONPATH=. python tools/phase18_build_golden_batch.py --output-dir output/phase18_handoffs/golden-batch --seeds 7007001 7007002 7007003 7007004
!PYTHONPATH=. python tools/phase18_verify_golden_batch.py --manifest output/phase18_handoffs/golden-batch/manifest.json
!cat output/phase18_handoffs/golden-batch/manifest.json


## Generate candidate 1 through the production-equivalent batch path
Only after batch integrity and runtime readiness pass, this may download the open model weights on first use. `--limit 1` executes the first locked candidate only; it does not modify the four-candidate manifest.

In [ ]:
!PYTHONPATH=. python tools/phase18_flux2_batch_execute.py --manifest output/phase18_handoffs/golden-batch/manifest.json --limit 1 --generation-dir output/phase18_generated --proof-dir output/phase18_visual_proof --dtype bfloat16 --result output/phase18_visual_proof/first-candidate-execution.json


In [ ]:
import json
from pathlib import Path
from IPython.display import display, Image
report = json.loads(Path('output/phase18_visual_proof/first-candidate-execution.json').read_text())
assert report['candidate_count'] == 1 and report['execution_scope'] == 'partial'
proof = Path(report['candidates'][0]['png'])
assert proof.is_file(), 'No real visual proof PNG was generated'
print(proof)
display(Image(filename=str(proof)))


## Optional: generate all four deterministic candidates
Run this only after candidate 1 succeeds technically and the runtime remains healthy. Candidates execute sequentially to avoid VRAM contention.

In [ ]:
# Uncomment after the first candidate succeeds:
# !PYTHONPATH=. python tools/phase18_flux2_batch_execute.py --manifest output/phase18_handoffs/golden-batch/manifest.json --generation-dir output/phase18_generated --proof-dir output/phase18_visual_proof --dtype bfloat16 --result output/phase18_visual_proof/batch-execution.json


## Golden Visual review
After the full batch exists, build the review template. PUL7SAR will not infer quality scores automatically: visually inspect every PNG, fill the explicit 0–10 scores and hard blockers, then run the quality selector. The strict Golden floor is 8.5 weighted with core dimensions at least 8.0; 9.0+ is the elite target.

In [ ]:
# Uncomment after the full batch is generated:
# !PYTHONPATH=. python tools/phase18_build_golden_review_template.py --execution-report output/phase18_visual_proof/batch-execution.json --output output/phase18_visual_proof/golden-review.json
# After filling golden-review.json:
# !PYTHONPATH=. python tools/phase18_review_golden_batch.py --execution-report output/phase18_visual_proof/batch-execution.json --review output/phase18_visual_proof/golden-review.json --output output/phase18_visual_proof/golden-selection.json
